# Lucida Step 8 SDK Core Image Flow

This notebook demonstrates Step-08 synchronous SDK control from handshake through event polling.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "python").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "python"))

In [ ]:
from uuid import uuid4

from lucida_daemon import default_local_ipc_uri
from lucida_sdk import launch_or_connect, shutdown_local_daemon

In [ ]:
app_name = f"lucida-step8-notebook-{uuid4().hex[:8]}"
local_ipc_uri = default_local_ipc_uri(app_name=app_name)
client = launch_or_connect(local_ipc_uri=local_ipc_uri)
client.hello_response

In [ ]:
session = client.session_create(label="step8-notebook")
session_id = str(session["session_id"])
subscription = client.subscribe_events(
    session_id=session_id,
    topics=["*"],
)
session_id

In [ ]:
opened = client.dataset_open(
    session_id=session_id,
    uri="synthetic://image",
    read_only=True,
)
job_id = str(opened["job"]["job_id"])
job = client.wait_for_job(session_id=session_id, job_id=job_id)

dataset_id = None
for event in subscription.iter_events(limit=32, poll_interval_s=0.0, max_idle_polls=4):
    if event["event_type"] == "dataset.opened":
        dataset_id = str(event["payload"]["dataset_id"])
        break

if dataset_id is None:
    raise RuntimeError("dataset.opened event was not observed")

job

In [ ]:
layer = client.layer_add_image(session_id=session_id, dataset_id=dataset_id, name="image")
view = client.view_create(session_id=session_id, label="main")
view_id = str(view["view_id"])
layer_id = str(layer["layer_id"])

client.view_bind_layer(session_id=session_id, view_id=view_id, layer_id=layer_id)
client.view_set_axis_index(
    session_id=session_id,
    view_id=view_id,
    axis_index={"axis": "z", "index": 0},
)
client.camera_set_mode(session_id=session_id, view_id=view_id, mode="panzoom")
client.camera_set_pose(
    session_id=session_id,
    view_id=view_id,
    pose={
        "position": [0.0, 0.0, 2.0],
        "target": [0.0, 0.0, 0.0],
        "up": [0.0, 1.0, 0.0],
    },
)

{
    "layer_id": layer_id,
    "view_id": view_id,
}

In [ ]:
events = subscription.poll(limit=64)
client.session_close(session_id=session_id)
client.close()
shutdown_local_daemon(local_ipc_uri=local_ipc_uri)
events[:3]